## Introduzione
### LISTA PROGRAMMI DI SOSTITUZIONE (per simulazione what-if)

OUTPUT:
1. `ta_coll.whatif.output_lista_programmi_sostituzione` — tabella front-end (1 riga per programma, colonne UI)
2. `ta_coll.whatif.output_palinsesto_passato_enriched` — tabella back-end (feature one-hot encoded per il modello)

OBIETTIVO:
Costruire la lista dei programmi candidati alla sostituzione, derivata dallo **storico Auditel** (canali Rai 1/2/3). Ogni programma deve avere StoricoShare popolato.  
Ogni riga della tabella front-end espone: `titolo · canale · tipologia · genere · età · share storico (%)` con un ID univoco per programma.

FLOW:
1. Loading storico Auditel (`ta_coll.whatif.storico_programmi`) filtrato per canali RAI, esclusione fasce notturne
2. Aggregazione feature storiche per `programma_norm`: genere predominante, target genere, target età
3. Calcolo statistiche sullo share (media ponderata ultime occorrenze, min/max/last) tramite `compute_stats_last_occurrences`
4. Join feature → storico completo + drop righe senza StoricoShare (filtro predicibilità)
5. Deduplicazione: 1 riga per programma (ultima messa in onda)
6. Creazione feature aggiuntive (ShareResiduo, HighValueShare, durata) → salvataggio tabella back-end
7. Selezione/rinomina colonne UI, aggregazione moda per programma, filtro programmi non interessanti → salvataggio tabella front-end

## Config

### Install

In [0]:
%run ./00_utility

In [0]:
%run ../FASE1/00_utils

In [0]:
# MAGIC %pip install beautifulsoup4 unidecode --quiet

In [0]:
%pip install pandarallel

In [0]:
import re
import time
import requests
import warnings
import pandas as pd
import numpy as np

from bs4 import BeautifulSoup
from datetime import date, timedelta
from delta.tables import DeltaTable

from pyspark.sql import functions as F

from pandarallel import pandarallel

pandarallel.initialize(nb_workers = 16, progress_bar=False)
warnings.filterwarnings("ignore")

### Parameters

In [0]:
# ============================================================
# TABLES
# ============================================================
AUDITEL_TABLE = "ta_coll.whatif.storico_programmi"

TABLE_OUTPUT_UI = "ta_coll.whatif.output_lista_programmi_sostituzione"
TABLE_OUTPUT_BACKEND = "ta_coll.whatif.output_palinsesto_passato_enriched"

# ============================================================
# CANALI (solo RAI: sono i canali sostituibili a UI)
# ============================================================
CANALI_RAI = ["Rai 1", "Rai 2", "Rai 3"]

MAP_CANALE_TIVU_AUDITEL = {
    "Rai 1": "Rai 1",
    "Rai 2": "Rai 2",
    "Rai 3": "Rai 3",
}

CANALI_TARGET = ["Rai 1", "Rai 2", "Rai 3"]

# ============================================================
# CHIAVI DI AGGREGAZIONE
# ============================================================
GROUP_KEYS = ['programma_norm']

# ============================================================
# CUTOFF 7 GIORNI PER CALCOLO STORICO SHARE
# ============================================================
CUTOFF_DAYS = 7

# ============================================================
# PROGRAMMI NON RIPETIBILI/NON DI INTERESSE
# ============================================================
TIPOLOGIE_ESCLUSE = [
    "CONCERTO",
    "INFORMAZIONE PARLAMENTARE",
    "INTERRUZIONE",
    "LIRICA",
    "MANIFESTAZIONI",
    "PREVISIONI DEL TEMPO",
    "PROSSIMAMENTE",
    "RUBRICA SPORTIVA",
    "SANTA MESSA",
    "SPETTACOLO MUSICALE",
    "SPORT",
    "TELEGIORNALE",
    "TG SPORT"
]
KEYWORDS_EVENTO = [
    "anniversario", "camp.europeo", "camp.mondiale", "coppa davis", "coppa del mondo",
    "europeo", "giobileo", "giornata", "giubileo", "giro ditalia", "mondiali",
    "nations league", "olimpi", "qualif", "tg1", "tg2", "tg3", "tgr", "tour de france", "world cup"
]

### Mapping

In [0]:
# Colonne raw Auditel usate per stimare il target di eta'
ETA_COLS = {
    "15_24": "1st_Screen_LiveVOSDAL_Adulti_15_24",
    "25_34": "1st_Screen_LiveVOSDAL_Adulti_25_34",
    "35_44": "1st_Screen_LiveVOSDAL_Adulti_35_44",
    "45_54": "1st_Screen_LiveVOSDAL_Adulti_45_54",
    "55_64": "1st_Screen_LiveVOSDAL_Adulti_55_64",
    "65_69": "1st_Screen_LiveVOSDAL_Adulti_65_69",
    "70_74": "1st_Screen_LiveVOSDAL_Adulti_70_74",
    "75_plus": "1st_Screen_LiveVOSDAL_Adulti_75plus",
}

# Mappatura fascia eta' fine -> etichetta UI ("45_54" -> "45+")
MAP_ETA_LABEL = {
    "15_24": "15+",
    "25_34": "25+",
    "35_44": "35+",
    "45_54": "45+",
    "55_64": "55+",
    "65_69": "65+",
    "70_74": "70+",
    "75_plus": "75+",
}

# Mappatura target genere -> etichetta UI
MAP_GENERE_LABEL = {
    "Uomini": "Uomo",
    "Donne": "Donna",
    "Misto": "Misto",
}

## Loading palinsesto storico

Carichiamo lo storico da `AUDITEL_TABLE`, filtrato per canali RAI (`CANALI_TARGET`) ed escludendo le fasce notturne (0–6:59 e 24+).  

In [0]:
storico_df = read_df_programmi(
    table_name=AUDITEL_TABLE
)
storico_df = storico_df[storico_df["Canale"].isin(CANALI_TARGET)].copy()
storico_df["canale_norm"] = storico_df["Canale"]
storico_df = storico_df.drop(columns="Canale")

# Escludiamo le messe in onda notturne (mezzanotte - 6:59)
not_night = ~((storico_df["Ora"] < 7) | ((storico_df["Ora"] >= 24)))
n_dropped = (~not_night).sum()
storico_df = storico_df[not_night].copy()
print(f"Righe notturne rimosse: {n_dropped}")

print(f"storico_df.shape: {storico_df.shape} | {storico_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
display(storico_df.head(100))

## Aggregazione feature storiche (per programma)

Aggreghiamo le feature storiche a livello di `programma_norm`.  

In [0]:
GROUP_KEYS = ['programma_norm']

# ---------- GENERE / TIPOLOGIA (DES_GENERE_ESTESA_INT) ----------
agg_genere = (
    storico_df
    .groupby(GROUP_KEYS)["DES_GENERE_ESTESA_INT"]
    .agg(lambda x: x.value_counts().index[0] if len(x.dropna()) > 0 else None)
    .reset_index(name="genere_predominante")
)

print(f"agg_genere.shape: {agg_genere.shape}")
display(agg_genere.head(10))

# ---------- TARGET GENERE (Uomini / Donne / Misto) ----------
def compute_gender_target(group):
    uomini = group["1st_Screen_LiveVOSDAL_Uomini"].median()
    donne = group["1st_Screen_LiveVOSDAL_Donne"].median()
    if pd.isna(uomini) or pd.isna(donne):
        return None
    if abs(uomini - donne) < 5:
        return "Misto"
    return "Uomini" if uomini > donne else "Donne"
# ---------- TARGET ETA' ----------
def compute_age_target(group):
    medians = {label: group[col].median() for label, col in ETA_COLS.items()}
    medians = {k: v for k, v in medians.items() if pd.notna(v)}
    if not medians:
        return None
    return max(medians, key=medians.get)

demo_rows = []
for keys, group in storico_df.groupby(GROUP_KEYS):
    demo_rows.append({
        "programma_norm": keys,
        "target_genere": compute_gender_target(group),
        "target_eta": compute_age_target(group),
    })

demo_df = pd.DataFrame(demo_rows)
print(f"demo_df.shape: {demo_df.shape}")
display(demo_df.head(10))

## Calcolo statistiche su share (per programma)

`compute_stats_last_occurrences` viene applicata via `groupby('programma_norm').parallel_apply(...)` sull'intero storico (oltre i 7 giorni fa).  
Il risultato viene poi **deduplicato**: per ogni `programma_norm` si conserva solo il record più recente (per `Data`), producendo **1 riga per programma** con le statistiche di share aggregate (`StoricoShare`, `StoricoShareMax`, `StoricoShareMin`, `StoricoShareLast`, ecc.).

In [0]:
agg_stats_share = (
    storico_df
    .groupby(GROUP_KEYS)
    .parallel_apply(
        compute_stats_last_occurrences,
        suffix='',
        cutoff_days=CUTOFF_DAYS,
        cutoff_num_occurrences=10, # default
        time_tolerance=3 # default
    )
    .reset_index(drop=True)
)

# Teniamo il record piu' recente per ogni programma_norm
agg_stats_share['Data'] = pd.to_datetime(agg_stats_share['Data'])
agg_stats_share = agg_stats_share.sort_values('Data').drop_duplicates(subset='programma_norm', keep='last')

# Teniamo solo programma_norm + le colonne aggiunte da compute_stats_last_occurrences
storico_cols = set(storico_df.columns)
agg_stats_share_cols = [c for c in agg_stats_share.columns if c not in storico_cols]
agg_stats_share = agg_stats_share[['programma_norm'] + agg_stats_share_cols]

print(f"agg_stats_share.shape: {agg_stats_share.shape}")
display(agg_stats_share.head(100))

## Tabella completa (tutte le righe di storico + le informazioni calcolate)

In [0]:
features_df = (
    storico_df
    .merge(agg_genere, on=GROUP_KEYS, how="left")
    .merge(demo_df, on=GROUP_KEYS, how="left")
    .merge(agg_stats_share, on=GROUP_KEYS, how="left")
)

# Rimuoviamo le righe con StoricoShare Null (e.g. programmi con una sola messa in onda, con messe in onda in pochi giorni senza separazione di almeno 7 giorni, etc.)
print(f"features_df.shape (pre-dropna): {features_df.shape}")
features_df = features_df.dropna(subset='StoricoShare')

print(f"features_df.shape (post-dropna): {features_df.shape}")
display(features_df.head(100))

## Deduplicazione tabella: teniamo solo la riga più recente di ogni programma presente nello storico

In [0]:
# Per ciascun programma_norm teniamo solo la riga più recente (ultima data di messa in onda).
features_df['Data'] = pd.to_datetime(features_df['Data'])
most_recent_df = (
    features_df
    .sort_values('Data', ascending=False)
    .drop_duplicates(subset='programma_norm', keep='first')
    .reset_index(drop=True)
)

print(f"most_recent_df.shape: {most_recent_df.shape}")
display(most_recent_df.head(100))

## Creazione features su statistiche dello share (per programma)

In [0]:
# Trasformiamo il dataframe dei programmi in Spark
final_share_stats_df = spark.createDataFrame(most_recent_df)

# Calcoliamo le feature sui dati di share:
# - ShareResiduo: differenza tra Share e StoricoShare
# - HighValueShare: True se StoricoShare >= 0.12
# - Programma_durata: differenza tra ORA_FINE_TRX e ORA_INIZIO_TRX
final_share_stats_df = final_share_stats_df.withColumn(
    'ShareResiduo', F.col('Share') - F.col('StoricoShare')
)
final_share_stats_df = final_share_stats_df.withColumn(
    'HighValueShare', F.col('StoricoShare') >= 0.12
)
final_share_stats_df = final_share_stats_df.withColumn(
    'Programma_durata', F.col("ORA_FINE_TRX") - F.col("ORA_INIZIO_TRX")
)

# Convertiamo i valori massimi, minimi e ultimi share storici in delta rispetto allo StoricoShare
for c in ['StoricoShareMax', 'StoricoShareMin', 'StoricoShareLast']:
    final_share_stats_df = final_share_stats_df.withColumn(
        c, F.col(c) - F.col('StoricoShare')
    )

print(f"most_recent_df shape: ({final_share_stats_df.count()}, {len(final_share_stats_df.columns)})")

## Salvataggio tabella back-end

In [0]:
final_share_stats_df.write.mode("overwrite").saveAsTable(TABLE_OUTPUT_BACKEND)

print(f"Righe in tabella back-end: {final_share_stats_df.count()} | Colonne: {len(final_share_stats_df.columns)}")

In [0]:
%sql
SELECT * FROM ta_coll.whatif.output_palinsesto_passato_enriched
ORDER BY programma_norm

## Selezione e rinomina colonne (per allinearle all'UI)

In [0]:
# Mapping eta' e genere
map_eta_expr = F.create_map([F.lit(x) for x in sum(MAP_ETA_LABEL.items(), ())])
map_genere_expr = F.create_map([F.lit(x) for x in sum(MAP_GENERE_LABEL.items(), ())])

lista_programmi_ui_df = final_share_stats_df.withColumn(
    "eta",
    map_eta_expr.getItem(F.col("target_eta"))
)
lista_programmi_ui_df = lista_programmi_ui_df.withColumn(
    "genere",
    F.when(
        map_genere_expr.getItem(F.col("target_genere")).isNotNull(),
        map_genere_expr.getItem(F.col("target_genere"))
    ).otherwise(F.col("target_genere"))
)

# Share storico in percentuale (UI mostra es. 17.4%)
lista_programmi_ui_df = lista_programmi_ui_df.withColumn(
    "share_storico_pct",
    F.round(F.col("StoricoShare") * 100, 1)
)

lista_programmi_ui_df = (
    lista_programmi_ui_df
    .withColumnRenamed("Programma", "titolo")
    .withColumnRenamed("canale_norm", "canale")
    .withColumnRenamed("Data", "data")
    .withColumnRenamed("durata_minuti", "durata")
    .withColumnRenamed("genere_predominante", "tipologia")
)

display(lista_programmi_ui_df.limit(100))

In [0]:
# Per ogni programma_norm, teniamo la combinazione PIÙ FREQUENTE di canale, tipologia, genere, età — così da avere 1 sola riga per programma.

# Calcoliamo la moda
unique_prog_df = (
    lista_programmi_ui_df.groupBy("programma_norm")
    .agg(
        F.mode("titolo").alias("titolo"),
        F.mode("canale").alias("canale"),
        F.mode("tipologia").alias("tipologia"),
        F.mode("genere").alias("genere"),
        F.mode("eta").alias("eta"),
        F.first("share_storico_pct").alias("share_storico_pct")
    )
    .orderBy(F.col("share_storico_pct").desc())
)

# ID univoco: canale + programma_norm, spazi sostituiti da underscore
unique_prog_df = unique_prog_df.withColumn("ID", F.col("programma_norm"))

unique_prog_df = unique_prog_df.withColumn("ID", F.regexp_replace(F.col("ID"), " ", "_"))

unique_prog_df = unique_prog_df.withColumn("row_num", F.row_number().over(Window.orderBy(F.col("share_storico_pct").desc())))
unique_prog_df = unique_prog_df.drop("row_num")

unique_prog_df = unique_prog_df.select("programma_norm", "titolo", "canale", "tipologia", "genere", "eta", "share_storico_pct", "ID")

print(f"Titoli univoci: {unique_prog_df.count()}")
display(unique_prog_df.limit(100))

## Filtro programmi non interessanti

In [0]:
# Filtriamo l'output togliendo i programmi che non ha senso proporre come sostituti

# Filtro 1: tipologie escluse
filtro_tipologia = ~F.col("tipologia").isin(TIPOLOGIE_ESCLUSE)

# Filtro 2: keyword evento nel programma_norm
filtro_keywords = F.lit(True)
for kw in KEYWORDS_EVENTO:
    filtro_keywords = filtro_keywords & ~F.col("programma_norm").contains(kw)

# Applichiamo il filtro combinato
totale_pre = unique_prog_df.count()
output_df = unique_prog_df.filter(filtro_tipologia & filtro_keywords)
totale_post = output_df.count()

# Programmi esclusi
excluded_df = unique_prog_df.filter(~(filtro_tipologia & filtro_keywords))
print(f"Programmi esclusi: {excluded_df.count()}")
display(excluded_df.orderBy(F.col("share_storico_pct").desc()))

print(f"Programmi PRIMA del filtro: {totale_pre}")
print(f"Programmi DOPO il filtro:   {totale_post}")
display(output_df.orderBy(F.col("share_storico_pct").desc()))

## Salvataggio tabella front-end

In [0]:
output_df.write.mode('overwrite').saveAsTable(TABLE_OUTPUT_UI)

print(f"Righe in tabella front-end: {output_df.count()} | Colonne: {len(output_df.columns)}")

In [0]:
%sql
SELECT * FROM ta_coll.whatif.output_lista_programmi_sostituzione
ORDER BY programma_norm